# 👥 Customer Behavior & Lifetime Value
> **Author:** Binh Pham | Maven Fuzzy Factory

Customer segmentation, cohort retention, CLV distribution, and funnel analysis.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi':120, 'figure.figsize':(12,5),
                     'axes.titlesize':13, 'axes.titleweight':'bold'})

BASE  = '..'
PROC  = f'{BASE}/data/processed'
orders   = pd.read_csv(f'{PROC}/cleaned_orders.csv',     parse_dates=['created_at'])
sessions = pd.read_csv(f'{PROC}/cleaned_website_sessions.csv', parse_dates=['created_at'])
master   = pd.read_csv(f'{PROC}/master_orders.csv',      parse_dates=['created_at'])


## 1. Customer Lifetime Value Distribution

In [ ]:

clv = master.groupby('user_id').agg(
    total_orders=('order_id','nunique'),
    lifetime_rev=('price_usd','sum')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(clv['lifetime_rev'], bins=40, color='#3498db', edgecolor='white')
axes[0].axvline(clv['lifetime_rev'].median(), color='red', linestyle='--',
                label=f"Median: ${clv['lifetime_rev'].median():.2f}")
axes[0].axvline(clv['lifetime_rev'].mean(), color='orange', linestyle='--',
                label=f"Mean: ${clv['lifetime_rev'].mean():.2f}")
axes[0].set_xlabel('Lifetime Revenue ($)')
axes[0].set_ylabel('Customer Count')
axes[0].set_title('Customer Lifetime Value Distribution')
axes[0].legend()

order_counts = clv['total_orders'].value_counts().sort_index()
axes[1].bar(order_counts.index, order_counts.values, color='#27ae60')
axes[1].set_xlabel('Total Orders per Customer')
axes[1].set_ylabel('Number of Customers')
axes[1].set_title('Purchase Frequency Distribution')

plt.tight_layout()
plt.savefig(f'{BASE}/notebooks/img_clv_distribution.png', bbox_inches='tight')
plt.show()

print(f"Single-purchase customers: {(clv['total_orders']==1).sum():,} ({(clv['total_orders']==1).mean()*100:.1f}%)")
print(f"Repeat buyers:             {(clv['total_orders']>1).sum():,} ({(clv['total_orders']>1).mean()*100:.1f}%)")


## 2. Revenue Contribution: Top 20% vs Bottom 80% (Pareto)

In [ ]:

clv_sorted = clv.sort_values('lifetime_rev', ascending=False).reset_index(drop=True)
clv_sorted['cumulative_rev']   = clv_sorted['lifetime_rev'].cumsum()
clv_sorted['cumulative_rev_pct'] = clv_sorted['cumulative_rev'] / clv_sorted['lifetime_rev'].sum() * 100
clv_sorted['customer_pct']     = (clv_sorted.index + 1) / len(clv_sorted) * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(clv_sorted['customer_pct'], clv_sorted['cumulative_rev_pct'],
        color='#9b59b6', linewidth=2)
ax.axhline(80, color='red', linestyle='--', alpha=0.7, label='80% Revenue')
ax.fill_between(clv_sorted['customer_pct'], clv_sorted['cumulative_rev_pct'], alpha=0.2, color='#9b59b6')
ax.set_xlabel('Customers (%)')
ax.set_ylabel('Cumulative Revenue (%)')
ax.set_title('Pareto Chart: Customer Revenue Concentration')
ax.legend()
plt.tight_layout()
plt.savefig(f'{BASE}/notebooks/img_pareto.png', bbox_inches='tight')
plt.show()

top20_rev = clv_sorted[clv_sorted['customer_pct'] <= 20]['lifetime_rev'].sum()
print(f"Top 20% of customers → {top20_rev / clv['lifetime_rev'].sum()*100:.1f}% of revenue")


## 3. Cohort Retention Heatmap

In [ ]:

orders_sorted = orders.sort_values('created_at')
cohort = orders.groupby('user_id')['created_at'].min().dt.to_period('M').rename('cohort_month')
orders_ch = orders.join(cohort, on='user_id')
orders_ch['order_month'] = orders_ch['created_at'].dt.to_period('M')
orders_ch['periods']     = (orders_ch['order_month'] - orders_ch['cohort_month']).apply(lambda x: x.n)

cohort_data = orders_ch.groupby(['cohort_month','periods'])['user_id'].nunique().reset_index()
cohort_pivot = cohort_data.pivot(index='cohort_month', columns='periods', values='user_id')
cohort_pct   = cohort_pivot.divide(cohort_pivot[0], axis=0) * 100

fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(cohort_pct.iloc[:, :13], annot=True, fmt='.0f', cmap='YlOrRd_r',
            linewidths=0.5, ax=ax, cbar_kws={'label':'Retention %'})
ax.set_title('Monthly Cohort Retention Heatmap (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Months Since First Purchase')
ax.set_ylabel('Cohort Month')
plt.tight_layout()
plt.savefig(f'{BASE}/notebooks/img_cohort_heatmap.png', bbox_inches='tight')
plt.show()


## 4. CLV by Acquisition Channel

In [ ]:

ch_clv = master.groupby(['user_id','utm_source']).agg(
    lifetime_rev=('price_usd','sum'), orders=('order_id','nunique')
).reset_index()
ch_summary = ch_clv.groupby('utm_source').agg(
    avg_clv=('lifetime_rev','mean'), median_clv=('lifetime_rev','median'),
    customers=('user_id','nunique'), total_rev=('lifetime_rev','sum')
).round(2).reset_index().sort_values('avg_clv', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = sns.color_palette('Set2', len(ch_summary))
axes[0].bar(ch_summary['utm_source'], ch_summary['avg_clv'], color=colors)
axes[0].set_ylabel('Avg CLV ($)')
axes[0].set_title('Average Customer Lifetime Value by Channel')
for i, row in ch_summary.reset_index().iterrows():
    axes[0].text(i, row['avg_clv'] + 0.5, f"${row['avg_clv']:.1f}", ha='center', fontsize=9)

axes[1].bar(ch_summary['utm_source'], ch_summary['total_rev'], color=colors)
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda v,_: f'${v/1e6:.1f}M'))
axes[1].set_title('Total Revenue by Acquisition Channel')

plt.tight_layout()
plt.savefig(f'{BASE}/notebooks/img_clv_by_channel.png', bbox_inches='tight')
plt.show()
print(ch_summary.to_string(index=False))
